# Polluted-River Public Goods (Sequential MDP)

**Strand 3 — Sequential games with UCB.**

A Polluted-River cleanup MDP where two agents may eat apples (immediate
reward) or clean the river (immediate cost, future common benefit). The river
has a carrying capacity that two eaters can collectively exceed. Tracks
cooperation outcomes (fair cooperation, volunteer's dilemma, collapse) and
the "temptation gap" metric for ensemble agents.

Heavy training (~500 episodes × 150 steps × 3 scenarios). Requires `torch`;
install via `./setup.sh --with-torch`. Historically also run on Colab — the
final cell gracefully no-ops outside a Colab runtime.

Environment, Q-network, replay buffer, and agents live in
`src/polluted_river/`. This notebook is orchestration only.

## Imports

In [ ]:
import sys, pathlib

_repo_root = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "requirements.txt").exists()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import torch
import numpy as np

from src.polluted_river.env import PollutedRiverEnv
from src.polluted_river.agents import EnsembleAgent, EpsilonGreedyAgent, SoftmaxAgent

## Parameters

In [ ]:
# Environment Settings
MAP_SIZE = 20
WINDOW_SIZE = 5
N_AGENTS = 2
MAX_STEPS = 150

# Training Settings
NUM_MODELS = 5
GAMMA = 0.999           # Very high Gamma (horizon ~1000 steps)
LR = 5e-4
BATCH_SIZE = 64
BUFFER_SIZE = 10000
UCB_BETA = 5.0
MAX_EPISODES = 500
PRIOR_SCALE = 3.0       # Increased Scale to boost Curiosity/Variance
TARGET_UPDATE_FREQ = 10

# Ensemble UCB Settings (Directed Exploration)
UCB_BETA_START = 5.0
UCB_BETA_MIN = 0.1
UCB_BETA_DECAY = 0.995

# Epsilon Greedy Settings (Random Exploration)
EPSILON_START = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.96

# Softmax Settings (Boltzmann Exploration)
TEMP_START = 5.0
TEMP_MIN = 0.1
TEMP_DECAY = 0.96

# Env physics (matches the original public-goods study)
POLLUTION_PER_EAT = 0.05
NATURAL_DECAY = 0.025
CLEAN_REDUCTION = 0.10
CLEAN_COST = 0.05

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Factories

Per-agent factories that close over the parameters above so `run_scenario`
stays parameter-free.

In [ ]:
def make_env():
    return PollutedRiverEnv(
        map_size=MAP_SIZE,
        window_size=WINDOW_SIZE,
        n_agents=N_AGENTS,
        max_steps=MAX_STEPS,
        pollution_per_eat=POLLUTION_PER_EAT,
        natural_decay=NATURAL_DECAY,
        clean_reduction=CLEAN_REDUCTION,
        clean_cost=CLEAN_COST,
    )


def make_ensemble_agent(state_dim, action_dim, agent_id):
    return EnsembleAgent(
        state_dim, action_dim, agent_id, device,
        num_models=NUM_MODELS, lr=LR, gamma=GAMMA,
        batch_size=BATCH_SIZE, buffer_size=BUFFER_SIZE,
        target_update_freq=TARGET_UPDATE_FREQ,
        prior_bias=-2.0, prior_scale=PRIOR_SCALE,
        ucb_beta_start=UCB_BETA_START, ucb_beta_min=UCB_BETA_MIN,
        ucb_beta_decay=UCB_BETA_DECAY,
        use_maximax=False,
    )


def make_egreedy_agent(state_dim, action_dim, agent_id):
    return EpsilonGreedyAgent(
        state_dim, action_dim, agent_id, device,
        lr=LR, gamma=GAMMA, batch_size=BATCH_SIZE, buffer_size=BUFFER_SIZE,
        target_update_freq=TARGET_UPDATE_FREQ,
        epsilon_start=EPSILON_START, epsilon_min=EPSILON_MIN, epsilon_decay=EPSILON_DECAY,
    )


def make_softmax_agent(state_dim, action_dim, agent_id):
    return SoftmaxAgent(
        state_dim, action_dim, agent_id, device,
        lr=LR, gamma=GAMMA, batch_size=BATCH_SIZE, buffer_size=BUFFER_SIZE,
        target_update_freq=TARGET_UPDATE_FREQ,
        temp_start=TEMP_START, temp_min=TEMP_MIN, temp_decay=TEMP_DECAY,
    )

## Scenario runner

In [ ]:
def run_scenario(scenario_name, agents):
    env = make_env()
    print(f"\n{'='*60}")
    print(f"SCENARIO: {scenario_name}")
    print(f"Agents: {[a.name for a in agents]}")
    print(f"{'='*60}")

    for episode in range(MAX_EPISODES):
        obs_n = env.reset()
        for a in agents:
            if hasattr(a, 'reset_diagnostics'): a.reset_diagnostics()

        total_rewards = [0, 0]
        total_cleans = [0, 0]
        steps = 0
        done = False

        while not done:
            actions = []
            for i, agent in enumerate(agents):
                act, _ = agent.get_action(obs_n[i])
                actions.append(act)

            next_obs_n, rewards, done, cleans = env.step(actions)

            for i, agent in enumerate(agents):
                agent.buffer.push(obs_n[i], actions[i], rewards[i], next_obs_n[i], done)
                agent.update()
                total_rewards[i] += rewards[i]
                total_cleans[i] += cleans[i]

            obs_n = next_obs_n
            steps += 1

        for agent in agents:
            if hasattr(agent, 'decay_epsilon'): agent.decay_epsilon()
            elif hasattr(agent, 'decay_temperature'): agent.decay_temperature()
            elif hasattr(agent, 'decay_beta'): agent.decay_beta()

        if episode % 20 == 0:
            total_score = sum(total_rewards)
            c0, c1 = total_cleans
            print(f"Ep {episode:03d} | Rwd: {total_rewards[0]:.1f}, {total_rewards[1]:.1f} | Cleans: {c0}, {c1}")

            # Diagnostic for Ensemble only
            if hasattr(agents[0], 'action_stats'):
                stats = agents[0].action_stats
                print("   [Diagnostic Agent 0]")
                act_names = ["Left", "Right", "Eat", "Clean"]

                # --- NEW METRIC: Temptation Gap ---
                # Difference between Eat (Selfish) and Clean (Coop)
                q_eat = np.mean(stats[2]['q']) if stats[2]['q'] else 0
                q_clean = np.mean(stats[3]['q']) if stats[3]['q'] else 0
                gap = q_eat - q_clean

                for a_idx in range(4):
                    q_vals = stats[a_idx]['q']
                    std_vals = stats[a_idx]['std']
                    if len(q_vals) > 0:
                        avg_q = np.mean(q_vals)
                        avg_std = np.mean(std_vals)
                        print(f"    - {act_names[a_idx]:<5}: Avg Q={avg_q:6.2f} | Avg Std={avg_std:6.4f}")

                print(f"    >> Temptation Gap (Eat - Clean): {gap:.4f}")
                if hasattr(agents[0], 'beta'):
                    print(f"    >> Current UCB Beta: {agents[0].beta:.4f}")

            # Diagnostic for E-Greedy / Softmax
            elif hasattr(agents[0], 'epsilon'):
                print(f"    >> Current Epsilon: {agents[0].epsilon:.4f}")
            elif hasattr(agents[0], 'temperature'):
                print(f"    >> Current Temperature: {agents[0].temperature:.4f}")


            if total_score < 30:
                 print(f"   -> OUTCOME: High Pollution (Collapse)")
            elif total_score > 60:
                total_cleaning = c0 + c1
                if total_cleaning == 0:
                     print(f"   -> OUTCOME: Fragile Sustainability (No cleaning)")
                else:
                    share_0 = c0 / total_cleaning
                    if share_0 > 0.8:
                         print(f"   -> OUTCOME: Volunteer's Dilemma (Agent 0 Farmer)")
                    elif share_0 < 0.2:
                         print(f"   -> OUTCOME: Volunteer's Dilemma (Agent 1 Farmer)")
                    else:
                         print(f"   -> OUTCOME: Fair Cooperation")

    print(f"Finished {scenario_name}")

## Run scenarios

In [ ]:
env_dim = (WINDOW_SIZE * 2 + 1) * 2 + 2
act_dim = 4

# Scenario 1: Two Ensemble UCB Agents
agents_1 = [make_ensemble_agent(env_dim, act_dim, 0), make_ensemble_agent(env_dim, act_dim, 1)]
run_scenario("2x Ensemble UCB", agents_1)

# Scenario 2: Two E-Greedy Agents
agents_2 = [make_egreedy_agent(env_dim, act_dim, 0), make_egreedy_agent(env_dim, act_dim, 1)]
run_scenario("2x E-Greedy", agents_2)

# Scenario 3: Two Softmax Agents
agents_3 = [make_softmax_agent(env_dim, act_dim, 0), make_softmax_agent(env_dim, act_dim, 1)]
run_scenario("2x Softmax Agents", agents_3)

## Cleanup (Colab only)

Disconnects the Colab runtime. No-ops on local.

In [ ]:
try:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
except ImportError:
    pass